In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [2]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    x = x - np.max(x)
    exp = np.exp(x)
    return exp / np.sum(exp)

In [3]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.lr = lr
        self.W1 = np.random.randn(hidden_size, input_size) * 0.01
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(output_size, hidden_size) * 0.01
        self.b2 = np.zeros((output_size, 1))

    def forward(self, x):
        self.z1 = self.W1 @ x + self.b1
        self.a1 = relu(self.z1)
        self.z2 = self.W2 @ self.a1 + self.b2
        self.y_hat = softmax(self.z2)
        return self.y_hat

    def backward(self, x, y):
        dz2 = self.y_hat - y
        dW2 = dz2 @ self.a1.T
        db2 = dz2

        dz1 = (self.W2.T @ dz2) * relu_derivative(self.z1)
        dW1 = dz1 @ x.T
        db1 = dz1

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

In [4]:
def load_images(path):
    with open(path, 'rb') as f:
        magic, num, rows, cols = np.frombuffer(f.read(16), dtype=np.uint32).byteswap()
        images = np.frombuffer(f.read(), dtype=np.uint8)
        return images.reshape(num, rows*cols)

def load_labels(path):
    with open(path, 'rb') as f:
        magic, num = np.frombuffer(f.read(8), dtype=np.uint32).byteswap()
        return np.frombuffer(f.read(), dtype=np.uint8)

X_train = load_images("mnist/train-images.idx3-ubyte")
y_train = load_labels("mnist/train-labels.idx1-ubyte")

X_test = load_images("mnist/t10k-images.idx3-ubyte")
y_test = load_labels("mnist/t10k-labels.idx1-ubyte")

X_train = X_train.reshape(-1, 784) / 255.0
X_test  = X_test.reshape(-1, 784) / 255.0

Y_train = np.zeros((y_train.size, 10))
Y_train[np.arange(y_train.size), y_train] = 1

Y_test = np.zeros((y_test.size, 10))
Y_test[np.arange(y_test.size), y_test] = 1


In [5]:
nn = NeuralNetwork(784, 128, 10, lr=0.01)

for epoch in range(11):
    correct = 0
    for i in range(2000):   # partial training for speed
        x = X_train[i].reshape(-1,1)
        y = Y_train[i].reshape(-1,1)

        out = nn.forward(x)
        nn.backward(x, y)

        if np.argmax(out) == np.argmax(y):
            correct += 1

    print(f"Epoch {epoch+1} Accuracy: {correct/2000*100}")

Epoch 1 Accuracy: 62.64999999999999
Epoch 2 Accuracy: 86.45
Epoch 3 Accuracy: 90.0
Epoch 4 Accuracy: 92.0
Epoch 5 Accuracy: 93.7
Epoch 6 Accuracy: 95.8
Epoch 7 Accuracy: 97.45
Epoch 8 Accuracy: 98.0
Epoch 9 Accuracy: 98.8
Epoch 10 Accuracy: 99.3
Epoch 11 Accuracy: 99.55000000000001


In [ ]:
import cv2

idx = np.random.randint(0, len(X_test))

img = X_test[idx].reshape(28,28)
label = np.argmax(Y_test[idx])

cv2.imshow("MNIST Image", img)
print("Label:", label)

cv2.waitKey(0)
cv2.destroyAllWindows()
